<a href="https://colab.research.google.com/github/meghanavanamala/predictiveanalysis/blob/recomment_product_tousers/recommand_products_users.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# Step 1: Sample Data
# -----------------------------

# User-product purchase matrix (1 = purchased, 0 = not purchased)
purchase_data = pd.DataFrame({
    'Product1': [1, 0, 1, 1, 0],
    'Product2': [0, 1, 1, 0, 1],
    'Product3': [1, 1, 1, 1, 0],
    'Product4': [0, 1, 0, 0, 1],
    'Product5': [1, 0, 0, 1, 0],
}, index=['User1', 'User2', 'User3', 'User4', 'User5'])

# User interests (tags for each user)
user_interests = {
    'User1': {'tech', 'gaming'},
    'User2': {'fashion', 'beauty'},
    'User3': {'tech', 'photography'},
    'User4': {'gaming', 'fitness'},
    'User5': {'fashion', 'tech'},
}

# Tags for each product
product_tags = {
    'Product1': {'tech'},
    'Product2': {'fashion'},
    'Product3': {'tech', 'photography'},
    'Product4': {'beauty'},
    'Product5': {'gaming', 'fitness'}
}

# -----------------------------
# Step 2: Recommendation Function
# -----------------------------

def recommend_products(target_user, top_n=3):
    # Step 1: User similarity matrix
    similarity_matrix = cosine_similarity(purchase_data)
    similarity_df = pd.DataFrame(similarity_matrix, index=purchase_data.index, columns=purchase_data.index)

    # Step 2: Similar users (excluding the user themself)
    similar_users = similarity_df[target_user].drop(target_user).sort_values(ascending=False)

    # Step 3: Weighted product score from similar users
    weighted_scores = pd.Series(0, index=purchase_data.columns, dtype='float64')
    for user, sim_score in similar_users.items():
        weighted_scores += purchase_data.loc[user] * sim_score

    # Step 4: Remove already purchased products
    already_purchased = purchase_data.loc[target_user][purchase_data.loc[target_user] > 0].index
    weighted_scores = weighted_scores.drop(already_purchased, errors='ignore')

    # Step 5: Filter products based on user's interests
    target_tags = user_interests.get(target_user, set())
    filtered_scores = {
        product: score for product, score in weighted_scores.items()
        if len(product_tags.get(product, set()) & target_tags) > 0
    }

    # Step 6: Fallback to general similar items if no interest match
    if not filtered_scores:
        print("No matches with interests. Showing similar items based on behavior...")
        filtered_scores = weighted_scores.to_dict()

    # Step 7: Return top-N recommendations
    recommended = pd.Series(filtered_scores).sort_values(ascending=False).head(top_n)
    return recommended

# -----------------------------
# Step 3: Test the Recommender
# -----------------------------

user_to_recommend = 'User1'
recommendations = recommend_products(user_to_recommend, top_n=3)

print(f"\n🎯 Recommendations for {user_to_recommend}:")
print(recommendations)


No matches with interests. Showing similar items based on behavior...

🎯 Recommendations for User1:
Product2    1.000000
Product4    0.333333
dtype: float64
